**DSCI2022 - HPC, Cloud, Quantum**
> **Professor: Dr. Sthefanie Passo**

> **E-mail: passo@datascience.msstate.edu**

# CPU Scheduling — Build Six Schedulers From Scratch

**DSCI 2022 · Data Science Lab · Module 1, Part III**

---

In the lecture we said the scheduler is the software that *creates a queue for jobs,
assigns resources to processes and threads, and decides who runs next.* We also said
something that should have bothered you:

> **Thread execution order is not guaranteed. The same program can produce a
> different order on every run.**

That sentence is doing a lot of work. The reason order is unpredictable is that a
**scheduling algorithm** you did not write is making the decision for you. This
notebook removes the mystery: you will implement six of those algorithms yourself
and watch exactly how each one reorders the same workload.

### What you will build

| Scheduler | Preemptive? | One-line idea |
|---|---|---|
| First-Come, First-Served | No | Whoever asked first, runs first |
| Shortest Job Next | No | Run the quickest job available |
| Priority | No | Run the most important job available |
| Shortest Remaining Time | **Yes** | Preemptive Shortest Job Next |
| Round Robin | **Yes** | Everyone gets a fixed time slice |
| Multilevel Queue | **Yes** | Separate queues, each with its own policy |

> **A preemptive algorithm is a scheduling policy where the operating system can interrupt a running process and force it back into the ready queue before that process finishes or uses up its allocated time.**

### How to use this notebook

Run the cells in order. Each scheduler gets three cells: an **explanation**, the
**implementation**, and a **run + what to notice**. At the end there is an
assignment where you design your own scheduler.

> **A note on what we are simulating.** We are *not* using Python's `threading`
> module here. Real threads are scheduled by your operating system, and you cannot
> tell the OS to use Shortest Job Next. To study the algorithms themselves we build
> a **discrete-time simulator**: a fake clock that ticks 0, 1, 2, 3… and asks our
> scheduler "who runs now?" at every tick. This is how scheduling is taught and
> researched, because it makes the algorithm the only variable.

---
# Part 0 — The Machinery

Before any algorithm, we need three things: a way to describe a **process**, a way
to **measure** the outcome, and a **simulator** that runs the clock.

## 0.1 What is a process, for our purposes?

A real process has code, data, state, memory, file handles. For scheduling, only
four numbers matter:

- **`arrival`** — the tick at which the process shows up and joins the ready queue.
  Before this, the scheduler does not know it exists.
- **`burst`** — how many ticks of CPU time it needs in total.
- **`priority`** — a number used only by the Priority scheduler. **Lower = more
  important**, which is the Unix convention and surprises people constantly.
- **`queue`** — which queue level it belongs to, used only by Multilevel Queue.

In [ ]:
from dataclasses import dataclass, replace


@dataclass
class Process:
    """A job waiting for the CPU. Only what the scheduler needs to decide."""
    # name, e.g. "P1"
    # tick at which it enters the ready queue
    # total CPU ticks required
    # lower number = higher priority (Unix convention)
    # queue level, for Multilevel Queue only

    def __repr__(self):
        return f"<{self.pid} arr={self.arrival} burst={self.burst}>"


# The workload every scheduler in this notebook will be given.
# Same four jobs every time, so any difference you see is caused by the ALGORITHM.
WORKLOAD = [
    Process("compleate the code")
]

for p in WORKLOAD:
    print(p)

<P1 arr=0 burst=7>
<P2 arr=2 burst=4>
<P3 arr=4 burst=1>
<P4 arr=5 burst=4>


## 0.2 How do we measure a scheduler?

A scheduler cannot make the total work smaller — those four jobs need 7+4+1+4 = 16
ticks of CPU no matter what. **All a scheduler can change is the order**, and
therefore *who waits, and how long*.

Four standard measurements, all derived from two timestamps:

| Metric | Formula | Answers |
|---|---|---|
| **Turnaround** | `finish − arrival` | Total time from showing up to being done |
| **Waiting** | `turnaround − burst` | Time spent in the queue doing nothing |
| **Response** | `first_start − arrival` | How long until it *first* got the CPU |

**Waiting time** is the classic "efficiency" number. **Response time** is the one
users feel — it is how long the cursor blinks before your keystroke appears. A
scheduler that is great at one is often bad at the other, and that tension is the
whole subject.

In [ ]:
@dataclass
class Result:
    """What happened to one process, once the simulation is over."""
    pid: str
    arrival: int
    burst: int
    start: int        # tick at which it FIRST got the CPU
    finish: int       # tick at which it completed

    @property
    def turnaround(self):
        return self.finish - self.arrival

    @property
    def waiting(self):
        return self.turnaround - self.burst

    @property
    def response(self):
        return self.start - self.arrival

## 0.3 The simulator

This is the engine, and it is the same for every algorithm. Read it once — you will
subclass against it all notebook.

Each tick, the engine does five things:

1. **Admit arrivals** — any process whose `arrival` has come joins the ready queue.
2. **Re-queue a preempted process** — one that was kicked off last tick rejoins,
   *behind* the new arrivals.
3. **Ask the scheduler** — `pick()` returns who runs this tick. This is the *only*
   part that changes between algorithms.
4. **Run one tick** — decrement remaining time, record it on the timeline.
5. **Check for completion or quantum expiry.**

The `pick()` method receives:

- `pool` — every process that could run right now (ready queue + whoever is running)
- `running` — who ran last tick, or `None`
- `clock` — the current tick
- `remaining` — dict of `pid → ticks still needed`

**The scheduler's entire job is to return one element of `pool`.** That is the
whole interface. Every algorithm below is a different one-line answer to
"which element?".

In [ ]:
class Scheduler:
    """Base class. A scheduling policy is just a rule for choosing from `pool`."""

    name = "Scheduler"
    preemptive = False   # True -> pick() is consulted EVERY tick, not just when idle

    def pick(self, pool, running, clock, remaining):
        """Return the process from `pool` that should run this tick."""
        raise NotImplementedError

    def on_dispatch(self, proc, clock):
        """Called when a process is given the CPU after not having it."""

    def on_tick(self, proc, clock):
        """Called after every tick this process runs."""

    def quantum_expired(self, proc, clock):
        """Return True to force the running process back into the queue."""
        return False


def run(processes, scheduler, max_ticks=10_000):
    """Simulate `processes` under `scheduler`. Returns (results, timeline)."""
    procs = [replace(p) for p in processes]          # never mutate the caller's list
    remaining = {p.pid: p.burst for p in procs}
    started, finished_at = {}, {}
    not_arrived = sorted(procs, key=lambda p: (p.arrival, p.pid))
    ready, timeline = [], []
    running, requeue, clock, done = None, None, 0, 0

    while done < len(procs):
        if clock > max_ticks:
            raise RuntimeError("Simulation never terminated — is your pick() starving someone?")

        # 1. admit every process that has arrived by now
        while not_arrived and not_arrived[0].arrival <= clock:
            ready.append(not_arrived.pop(0))
        # 2. a process preempted last tick rejoins BEHIND the new arrivals
        if requeue is not None:
            ready.append(requeue)
            requeue = None

        # 3. the scheduling decision
        if running is None or scheduler.preemptive:
            pool = ready + ([running] if running is not None else [])
            choice = scheduler.pick(pool, running, clock, remaining) if pool else None
            if choice is not running:
                if running is not None:
                    ready.append(running)
                if choice is not None and choice in ready:
                    ready.remove(choice)
                running = choice
                if running is not None:
                    scheduler.on_dispatch(running, clock)

        # 4a. nobody to run -> the CPU idles for this tick
        if running is None:
            timeline.append(("idle", clock, clock + 1))
            clock += 1
            continue

        # 4b. run exactly one tick
        started.setdefault(running.pid, clock)
        remaining[running.pid] -= 1
        if timeline and timeline[-1][0] == running.pid and timeline[-1][2] == clock:
            timeline[-1] = (running.pid, timeline[-1][1], clock + 1)   # extend the segment
        else:
            timeline.append((running.pid, clock, clock + 1))
        clock += 1
        scheduler.on_tick(running, clock)

        # 5. finished, or time slice used up?
        if remaining[running.pid] == 0:
            finished_at[running.pid] = clock
            done += 1
            running = None
        elif scheduler.quantum_expired(running, clock):
            requeue = running
            running = None

    results = [Result(p.pid, p.arrival, p.burst, started[p.pid], finished_at[p.pid])
               for p in sorted(procs, key=lambda p: p.pid)]
    return results, timeline


print("Engine ready.")

## 0.4 Seeing the result

Numbers in a table are hard to feel. A **Gantt chart** shows the CPU as a strip of
time with each process coloured in, and it makes the personality of each algorithm
obvious at a glance.

In [ ]:
def gantt(timeline, width=3):
    """Render the CPU timeline as text. Works everywhere, no dependencies."""
    merged = []
    for pid, s, e in timeline:
        if merged and merged[-1][0] == pid and merged[-1][2] == s:
            merged[-1] = (pid, merged[-1][1], e)
        else:
            merged.append((pid, s, e))
    bar = axis = ""
    for pid, s, e in merged:
        cells = (e - s) * width
        bar += "|" + pid.center(cells - 1)
        axis += "|" + str(s).ljust(cells - 1)
    return bar + "|\n" + axis + "|" + str(merged[-1][2])


def report(results, timeline, title):
    """Print the Gantt chart and the metrics table for one simulation."""
    print(f"\n{title}")
    print("=" * 64)
    print(gantt(timeline))
    print()
    print(f"{'PID':<5}{'Arr':>5}{'Burst':>7}{'Start':>7}{'End':>6}"
          f"{'Turn':>7}{'Wait':>7}{'Resp':>7}")
    print("-" * 64)
    for r in results:
        print(f"{r.pid:<5}{r.arrival:>5}{r.burst:>7}{r.start:>7}{r.finish:>6}"
              f"{r.turnaround:>7}{r.waiting:>7}{r.response:>7}")
    n = len(results)
    print("-" * 64)
    print(f"{'avg':<5}{'':>5}{'':>7}{'':>7}{'':>6}"
          f"{sum(r.turnaround for r in results)/n:>7.2f}"
          f"{sum(r.waiting for r in results)/n:>7.2f}"
          f"{sum(r.response for r in results)/n:>7.2f}")


def averages(results):
    n = len(results)
    return (sum(r.turnaround for r in results) / n,
            sum(r.waiting for r in results) / n,
            sum(r.response for r in results) / n)


print("Reporting ready.")

---
# 1 · First-Come, First-Served (FCFS)

## The idea

The queue at a coffee shop. Whoever arrived first gets served first, and once you
start being served nobody can cut in. **Non-preemptive**: a running process keeps
the CPU until it finishes.

This is the only scheduler that is obviously *fair* in the everyday sense of the
word, and it is the easiest to implement — the rule is "sort by arrival time".

## The catch: the convoy effect

If one enormous job arrives first, every short job behind it waits for the whole
thing. One slow truck on a single-lane road holds up a convoy of fast cars. Watch
`P3` below: it needs **one** tick of CPU and waits **seven** for it.

In [ ]:



results, timeline = run(WORKLOAD, FCFS())
report(results, timeline, "FCFS")

### What to notice

- **The Gantt chart is just the arrival order**: P1, P2, P3, P4. No surprises — and
  no cleverness either.
- **P3 is the victim.** It is a 1-tick job that waits 7 ticks. Its turnaround is 8×
  its actual work. That ratio is the convoy effect in one number.
- **Waiting and response time are identical for every process.** That is a signature
  of every non-preemptive scheduler: once you start, you finish, so the moment you
  first run is the only moment you run.

FCFS is the baseline every other algorithm is trying to beat. Write down
**avg waiting = 4.75** and let us see who can do better.

---
# 2 · Shortest Job Next (SJN)

## The idea

Also called Shortest Job First. When the CPU comes free, look at everything waiting
and run **the one with the smallest burst time**. Still non-preemptive: a job that
starts, finishes.

## Why it is special

SJN is **provably optimal** for average waiting time among non-preemptive
algorithms. Not "good in practice" — provably the best possible. The intuition:
putting a short job ahead of a long one saves the short job a lot of waiting and
costs the long job only a little, so the trade is always favourable.

## Why we cannot actually use it

**It requires knowing the future.** To run the shortest job, you must know how long
each job will take *before you run it*. Real operating systems do not know this.
They estimate it from past behaviour (exponential averaging of previous bursts),
which is a guess, and guesses are wrong.

So SJN is best understood as a **lower bound** — the score to measure real
schedulers against, not a thing you can deploy.

In [ ]:


results, timeline = run(WORKLOAD, ShortestJobNext())
report(results, timeline, "Shortest Job Next")

### What to notice

- **P3 jumped the queue.** At tick 7, P2 (burst 4), P3 (burst 1) and P4 (burst 4)
  are all waiting. P3 is shortest, so it runs — and finishes in a single tick.
  Its waiting time drops from **7 → 3**.
- **Average waiting fell from 4.75 → 4.00.** No process worked harder; we only
  changed the order.
- **P1 still ran first and still hogged 7 ticks.** SJN cannot fix that, because it
  is non-preemptive and P1 was the only job at tick 0. Watch what SRT does about
  this in section 4.
- **P2 got slightly worse** (waiting 5 → 6). Optimal *on average* does not mean
  better for everyone. Somebody always pays.

> **Starvation warning.** If short jobs keep arriving, a long job may never be the
> shortest and never run. SJN can starve long jobs indefinitely.

---
# 3 · Priority Scheduling

## The idea

Every process carries a priority number. Run the most important one available.
**In our convention (and Unix's), lower number = higher priority** — priority 1
beats priority 3. This trips up nearly everyone the first time.

SJN is actually a special case of Priority scheduling where `priority = burst`.

## Where priorities come from

Not from nowhere. Real systems set them from: how much memory a job needs, whether
it is interactive or batch, who submitted it (on a shared cluster like Ptolemy,
paying groups get better priority), and whether it is a system process or a user
process.

## The catch: starvation

A low-priority process can wait forever if high-priority work keeps arriving. This
is not theoretical — **MIT's IBM 7094 was shut down in 1973 and operators found a
job from 1967 still waiting to run.**

The standard fix is **ageing**: gradually raise the priority of any process that has
been waiting a long time, so everything eventually reaches the front. You will
implement ageing in the assignment.

In [ ]:


results, timeline = run(WORKLOAD, PriorityScheduler())
report(results, timeline, "Priority — lower number means more important")

print("\nPriorities:", {p.pid: p.priority for p in WORKLOAD})

### What to notice

- **P3 went from hero to victim.** Under SJN it ran first at tick 7; here it has the
  worst priority (3) and is pushed to the very end. It waits **11 ticks for 1 tick
  of work.**
- **Average waiting got worse than FCFS** (5.50 vs 4.75). Priority scheduling is not
  trying to minimise waiting time — it is trying to honour importance. If your
  priorities do not reflect anything real, you get the cost with none of the benefit.
- **P2 (priority 1) ran as soon as the CPU was free.** That is the algorithm doing
  exactly what it was told.

This is the clearest example in the notebook that **"best" depends entirely on what
you are optimising for.**

---
# 4 · Shortest Remaining Time (SRT)

## The idea

Preemptive Shortest Job Next. Same rule — run the shortest — but now we re-decide
**every single tick**, and we compare **remaining** time rather than total burst.

The consequence: if a new job arrives whose burst is shorter than what is left of
the running job, the running job is **kicked off the CPU mid-execution.**

## The one-line difference

```python
preemptive = True                        # ask me every tick, not just when idle
key = remaining[p.pid]                   # what's LEFT, not the original burst
```

That is genuinely the entire difference from SJN. Everything else — the dramatic
change you are about to see in the Gantt chart — falls out of those two lines.

In [ ]:


results, timeline = run(WORKLOAD, ShortestRemainingTime())
report(results, timeline, "Shortest Remaining Time (preemptive SJN)")

### What to notice

- **P1 got interrupted twice.** It starts at tick 0, but at tick 2 P2 arrives with
  burst 4 against P1's 5 remaining — so P1 is thrown off. At tick 4, P3 arrives with
  burst 1 and throws P2 off. P1 does not finish until tick 16.
- **Look at the response column: 0, 0, 0, 2 — average 0.5.** Under FCFS it was 4.75.
  Every job except P4 got the CPU essentially the instant it arrived. This is why
  preemption exists: it is what makes a system feel *responsive*.
- **Average waiting is the best we have seen: 3.00.** SRT beats plain SJN because it
  can act on information that arrives *after* a job has started.
- **P1 paid for everyone else.** Its waiting time is 9 — the worst single number in
  the notebook. Preemption redistributes pain toward long jobs.

> **The hidden cost.** Every preemption is a **context switch**: save P1's registers
> and program counter, load P2's. Our simulator pretends this is free. On real
> hardware it costs roughly 1–10 microseconds, and a scheduler that switches too
> eagerly can spend more time switching than working. You will measure this in the
> assignment.

---
# 5 · Round Robin (RR)

## The idea

The scheduler of every interactive system you have ever used. Each process gets a
fixed **time quantum** (say 2 ticks). When your slice is up, you go to the **back of
the queue** and the next process runs. Repeat forever.

No process can monopolise the CPU, and nothing can starve — with *n* processes and
quantum *q*, you are guaranteed the CPU within *(n−1) × q* ticks. That guarantee is
the whole point.

## The quantum is the entire design decision

| Quantum | Behaviour |
|---|---|
| **Too small** | Superb response time, but you spend all your time context switching |
| **Too large** | Context switching becomes free — because you have degenerated into FCFS |

The classic rule of thumb: **80% of CPU bursts should be shorter than the quantum.**
Real Linux uses a target latency of a few milliseconds rather than a fixed quantum,
but the tension is the same.

We run it twice below — `q=2` and `q=4` — so you can see the trade directly.

In [ ]:


if __name__ == "__main__":
    WORKLOAD = [
        Process("P1", arrival=0, burst=7, priority=2),
        Process("P2", arrival=2, burst=4, priority=1),
        Process("P3", arrival=4, burst=1, priority=3),
        Process("P4", arrival=5, burst=4, priority=2),
    ]
    for q in (2, 4):
        results, timeline = run(WORKLOAD, RoundRobin(quantum=q))
        report(results, timeline, f"Round Robin, quantum = {q}")

### What to notice

- **The Gantt chart is shredded.** With `q=2` the CPU changes hands 9 times. Compare
  that to FCFS's 4 clean blocks. Every one of those boundaries is a context switch
  you are paying for.
- **Response time is excellent: 1.50 with `q=2`.** Second only to SRT — and unlike
  SRT, Round Robin needs **no knowledge of the future whatsoever.** It never asks how
  long a job will take. That is why it is deployable and SJN is not.
- **Average waiting is 5.00 — worse than FCFS.** Round Robin is *not* trying to
  minimise waiting time. It is buying fairness and responsiveness, and it pays for
  them in turnaround.
- **Raising the quantum to 4 moves every number toward FCFS**: waiting improves
  (5.00 → 4.50), response degrades (1.50 → 3.25), and the chart gets blockier. Push
  the quantum past the longest burst and you would get FCFS exactly.

Try `q=1` and `q=20` in the cell above and watch the two extremes.

---
# 6 · Multilevel Queue

## The idea

Not one queue but several, each holding a different *class* of work and each running
its **own algorithm**. A typical split:

- **Queue 0 — foreground / interactive.** Your editor, your terminal. Needs fast
  response, so it uses **Round Robin**.
- **Queue 1 — background / batch.** Log processing, nightly builds, your Ptolemy job.
  Nobody is watching, so throughput matters more than response: **FCFS**.

The rule between queues is **strict priority**: nothing in queue 1 runs while
anything in queue 0 is ready. A queue-0 arrival **preempts** a running queue-1 job.

This is closest to what real operating systems actually do — and it is what Slurm is
doing when it sorts your Ptolemy jobs into partitions.

## The catch

Queue 1 can starve completely under sustained interactive load, and a process is
stuck in whichever queue it was born in. The fix is a **Multilevel *Feedback*
Queue**, which lets processes migrate between levels based on observed behaviour —
that is the stretch goal in the assignment.

In [ ]:
class MultilevelQueue(Scheduler):
    """Strict priority between queues; each queue runs its own policy.

    Queue 0 (interactive) -> Round Robin.
    Queue 1 (batch)       -> FCFS, and only when queue 0 is empty.
    """
    preemptive = True     # so a queue-0 arrival can preempt a running queue-1 job

    def __init__(self, quantum=2):
        self.quantum = quantum
        self.used = 0
        self.name = f"Multilevel Queue (Q0 = RR q={quantum}, Q1 = FCFS)"

    def pick(self, pool, running, clock, remaining):
        # Compleate
        return 0
    
    def on_dispatch(self, proc, clock):
        self.used = 0

    def on_tick(self, proc, clock):
        self.used += 1

    def quantum_expired(self, proc, clock):
        return proc.queue == 0 and self.used >= self.quantum


# A workload with two classes of job: A and C are interactive, B and D are batch.
MIXED = [
    Process("A", arrival=0, burst=4, queue=0),   # interactive
    Process("B", arrival=1, burst=3, queue=1),   # batch
    Process("C", arrival=2, burst=3, queue=0),   # interactive
    Process("D", arrival=3, burst=2, queue=1),   # batch
]

results, timeline = run(MIXED, MultilevelQueue(quantum=2))
report(results, timeline, "Multilevel Queue")
print("\nQueue assignment:", {p.pid: ("interactive" if p.queue == 0 else "batch")
                              for p in MIXED})

### What to notice

- **B arrived at tick 1 and did not start until tick 7.** It is a batch job, and
  every interactive job in the system goes ahead of it — even C, which arrived *a
  tick later* than B. Queue level beats arrival time, always.
- **A and C interleave** (A, C, A, C) because within queue 0 the policy is Round
  Robin. Once queue 0 empties at tick 7, B and D run back-to-back in arrival order,
  because queue 1's policy is FCFS.
- **The interactive jobs both have response time 0.** That is the design goal, and it
  is achieved entirely at the batch jobs' expense.
- **Now imagine interactive jobs keep arriving.** B and D never run. That is the
  starvation this design invites, and why feedback queues exist.

---
# 7 · Head to Head

Six algorithms, one workload. The total CPU work was identical in every run — 16
ticks. Everything below is a consequence of **ordering alone**.

In [ ]:
WORKLOAD = [
        Process("P1", arrival=0, burst=7, priority=2),
        Process("P2", arrival=2, burst=4, priority=1),
        Process("P3", arrival=4, burst=1, priority=3),
        Process("P4", arrival=5, burst=4, priority=2),
    ]

schedulers = [FCFS(), ShortestJobNext(), PriorityScheduler(),
              ShortestRemainingTime(), RoundRobin(2), RoundRobin(4)]

print(f"{'Scheduler':<32}{'Turnaround':>12}{'Waiting':>10}{'Response':>10}{'Switches':>10}")
print("=" * 74)

rows = []
for sched in schedulers:
    res, tl = run(WORKLOAD, sched)
    t, w, r = averages(res)
    switches = len([seg for seg in tl if seg[0] != "idle"]) - 1
    rows.append((sched.name, t, w, r, switches))
    print(f"{sched.name:<32}{t:>12.2f}{w:>10.2f}{r:>10.2f}{switches:>10}")

print("=" * 74)
best_w = min(rows, key=lambda x: x[2])
best_r = min(rows, key=lambda x: x[3])
print(f"Lowest average waiting  : {best_w[0]}  ({best_w[2]:.2f})")
print(f"Lowest average response : {best_r[0]}  ({best_r[3]:.2f})")

In [ ]:
# Switch the workload values and see how the schedulers perform differently. 


### Reading the table

**There is no winner, and that is the lesson.**

- **Shortest Remaining Time wins on both waiting and response** — and is unusable,
  because it needs to know burst times in advance. The best algorithm is the one you
  cannot have.
- **Round Robin has the most context switches by a wide margin.** Our simulator
  charges nothing for them. A real one would, and RR's numbers would get worse while
  FCFS's stayed put.
- **Priority is the worst on waiting time** — because it is not optimising waiting
  time. Judging it by that column is judging a truck by its lap time.
- **FCFS is mediocre at everything and excellent at one thing: being predictable.**
  It never starves anyone and never surprises you.

The question is never *"which scheduler is best?"* It is **"what am I optimising
for, and what am I willing to give up?"**

---
# 8 · Two Effects Worth Seeing Directly

The pathologies below are the reason schedulers are a subject rather than a
one-liner. Each cell constructs a workload designed to make one failure obvious.

In [ ]:
# ---- The convoy effect: one huge job arrives first ----
convoy = [
    Process("BIG", arrival=0, burst=20),
    Process("s1", arrival=1, burst=1),
    Process("s2", arrival=2, burst=1),
    Process("s3", arrival=3, burst=1),
]

for sched in [FCFS(), ShortestRemainingTime()]:
    res, tl = run(convoy, sched)
    t, w, r = averages(res)
    print(f"\n{sched.name}")
    print(gantt(tl))
    print(f"  avg waiting = {w:.2f}   avg response = {r:.2f}")
    for x in res:
        if x.pid.startswith("s"):
            print(f"    {x.pid}: needed 1 tick, waited {x.waiting}")

**The convoy effect.** Under FCFS the three one-tick jobs wait behind a 20-tick
job for work that takes them a single tick each. Shortest Remaining Time preempts
`BIG` the moment a short job appears and the short jobs finish almost immediately.

Same four jobs. Same 23 ticks of work. Wildly different experience for `s1`–`s3`.

In [ ]:
# ---- Starvation: low-priority work never reaching the CPU ----
starve = [Process("LOW", arrival=0, burst=3, priority=9)]
for i in range(6):                       # a steady stream of important work
    starve.append(Process(f"hi{i}", arrival=i, burst=2, priority=1))

res, tl = run(starve, PriorityScheduler())
print(gantt(tl))
low = [r for r in res if r.pid == "LOW"][0]
print(f"\nLOW arrived at tick {low.arrival}, "
      f"first ran at tick {low.start}, waited {low.waiting} ticks.")
print("Every high-priority job that arrived went ahead of it.")
print("\nWith arrivals that never stop, LOW would never run at all.")

**Starvation.** `LOW` arrived first and ran last. Our stream of high-priority
jobs eventually stops, so `LOW` does get its turn — but if the stream continued,
it would wait forever.

**The fix is ageing**, and implementing it is part of your assignment: increase a
process's effective priority the longer it has been waiting, so everything reaches
the front eventually.

---
---

# Part 9 · Your Assignment — Build Your Own Scheduler

You have seen six policies. Now you write your own.

## 9.1 The Rules

Your scheduler **must** obey the following contract. These are not style
suggestions — the validator in §9.3 checks most of them mechanically, and a
scheduler that breaks them is wrong even if its numbers look good.

---

**R1 — Subclass `Scheduler` and set a `name`.**
Your class inherits from `Scheduler` and defines a human-readable `name` attribute.
Do not modify the `Scheduler` base class or the `run()` engine. Everyone's scheduler
must run on the same unmodified simulator, or the comparison is meaningless.

**R2 — `pick()` must return an element of `pool`.**
Not a copy, not a new `Process`, not `None`, not a pid string — one of the actual
objects handed to you. Returning something else is the single most common bug in
this assignment. Use `is` to check identity if unsure.

**R3 — `pick()` must be deterministic.**
Given the same `pool`, `running`, `clock` and `remaining`, it must return the same
process every time. **Always break ties explicitly**, and make `pid` the final
tie-breaker:

```python
return min(pool, key=lambda p: (my_score(p), p.arrival, p.pid))
```

No `random`, no `set` iteration order, no dict ordering assumptions. If two students
run your scheduler they must get identical Gantt charts.

**R4 — Declare whether you are preemptive.**
Set `preemptive = True` if `pick()` should be consulted every tick, `False` if the
running process should keep the CPU until it finishes or its quantum expires. Wrong
value here silently produces a different algorithm from the one you designed.

**R5 — Do not mutate the inputs.**
`pick()` must not modify `pool`, the `Process` objects, or `remaining`. Read them,
score them, return one. Keep your own state in `self` (see `RoundRobin` for how).

**R6 — Be work-conserving.**
If `pool` is non-empty you must return a process. Never leave the CPU idle while
work is waiting. (If you want to argue for a non-work-conserving design, that is
interesting — but say so explicitly in your write-up and justify it.)

**R7 — Terminate.**
Every process must eventually finish. If your policy can starve a process forever,
either fix it or document precisely when it happens and why you accept it. The
engine raises `RuntimeError` after 10,000 ticks.

**R8 — Justify your tie-breaks and constants.**
Every magic number (a quantum, an ageing rate, a weight) needs one sentence saying
why that value. "I tried it and it worked" is acceptable *if* you show what else you
tried.

---

## 9.2 What To Build

Pick **one** of the following. Aim for a policy you can defend, not the most
complicated one you can write.

### Option A — Priority with Ageing *(recommended starting point)*
Fix the starvation you saw in §8. Start from `PriorityScheduler`, and lower a
process's effective priority number by 1 for every `k` ticks it has spent waiting.

- Choose `k` and justify it.
- Demonstrate it on the `starve` workload and show `LOW` no longer waits pathologically.
- Report what it costs: how much worse are the high-priority jobs?

### Option B — Highest Response Ratio Next (HRRN)
A classic non-preemptive policy that balances SJN against starvation. At each
decision, run the process with the highest ratio:

$$\text{ratio} = \frac{\text{waiting time} + \text{burst}}{\text{burst}}$$

Short jobs start with a high ratio; long jobs climb as they wait, so nothing starves.

- You will need to track waiting time yourself — `clock - p.arrival - (ticks already run)`.
- Compare against both SJN and FCFS on the standard `WORKLOAD`.

### Option C — Multilevel Feedback Queue
Extend `MultilevelQueue` so processes **move between levels**. A process that uses
its entire quantum is CPU-bound, so demote it; a process that has waited too long
gets promoted.

- Start with 3 levels, quanta 2 / 4 / 8, FCFS at the bottom.
- Show a workload where this beats plain Multilevel Queue.

### Option D — Your own design
Invent a policy for a scenario you care about — a fair-share scheduler for a shared
cluster, a deadline-aware scheduler, an energy-aware one that prefers batching. Must
be non-trivial and must obey R1–R8. **Clear your idea with me before you start.**

---

## 9.3 Self-Check

Run the validator below before submitting. It mechanically checks R2, R3, R5, R6 and
R7. **It does not check that your algorithm is correct** — only that it is a
well-behaved citizen of the engine. Passing is necessary, not sufficient.

In [ ]:
import copy


def validate_scheduler(scheduler_factory, verbose=True):
    """Check a scheduler against rules R2, R3, R5, R6, R7. Returns True if it passes."""
    problems = []

    tests = {
        "standard":  WORKLOAD,
        "all-at-once": [Process(f"X{i}", 0, 3, priority=i % 3) for i in range(5)],
        "idle gap":  [Process("A", 0, 2), Process("B", 10, 2)],
        "single":    [Process("solo", 0, 4)],
        "long tail": [Process("BIG", 0, 15)] + [Process(f"s{i}", i, 1) for i in range(1, 5)],
    }

    for label, work in tests.items():
        snapshot = copy.deepcopy(work)

        # --- R7: terminates ---
        try:
            res1, tl1 = run(work, scheduler_factory())
        except RuntimeError as e:
            problems.append(f"R7 [{label}]: did not terminate — {e}")
            continue
        except Exception as e:
            problems.append(f"[{label}]: raised {type(e).__name__}: {e}")
            continue

        # --- R5: inputs untouched ---
        if any(a != b for a, b in zip(work, snapshot)):
            problems.append(f"R5 [{label}]: the input processes were mutated")

        # --- R3: deterministic ---
        res2, tl2 = run(work, scheduler_factory())
        if tl1 != tl2:
            problems.append(f"R3 [{label}]: two identical runs produced different timelines")

        # --- R6: work-conserving ---
        arrivals = sorted(p.arrival for p in work)
        for pid, s, e in tl1:
            if pid == "idle":
                waiting_then = [a for a in arrivals if a <= s]
                finished_by = sum(1 for r in res1 if r.finish <= s)
                if len(waiting_then) > finished_by:
                    problems.append(f"R6 [{label}]: CPU idle at tick {s} with work available")
                    break

        # --- sanity: every process ran for exactly its burst ---
        for r in res1:
            ran = sum(e - s for pid, s, e in tl1 if pid == r.pid)
            if ran != r.burst:
                problems.append(f"[{label}]: {r.pid} ran {ran} ticks but needed {r.burst}")

    # --- R2: pick() returns a pool member ---
    class Probe(Scheduler):
        pass
    sched = scheduler_factory()
    original_pick = sched.pick
    bad = []

    def checked_pick(pool, running, clock, remaining):
        choice = original_pick(pool, running, clock, remaining)
        if not any(choice is p for p in pool):
            bad.append(clock)
        return choice

    sched.pick = checked_pick
    try:
        run(WORKLOAD, sched)
    except Exception as e:
        problems.append(f"R2 probe raised {type(e).__name__}: {e}")
    if bad:
        problems.append(f"R2: pick() returned a non-pool object at tick(s) {bad[:5]}")

    if verbose:
        if problems:
            print("FAILED\n")
            for p in problems:
                print(" ·", p)
        else:
            print("PASSED — R2, R3, R5, R6, R7 all satisfied.")
            print("(This does not check that your algorithm does what you intended.)")
    return not problems


# The six reference schedulers should all pass.
for factory in [FCFS, ShortestJobNext, PriorityScheduler, ShortestRemainingTime,
                lambda: RoundRobin(2), lambda: MultilevelQueue(2)]:
    name = factory().name
    ok = validate_scheduler(factory, verbose=False)
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

## 9.4 Starter Template

Copy this cell, rename the class, and fill in `pick()`.

In [ ]:
class MyScheduler(Scheduler):
    name = "TODO: name your policy"
    preemptive = False        # R4: True = re-decide every tick

    def __init__(self):
        # R5: keep any state you need here, never in the Process objects.
        pass

    def pick(self, pool, running, clock, remaining):
        """Return ONE element of `pool` (R2), deterministically (R3)."""
        # R3: always end the sort key with p.pid so ties resolve the same way twice.
        return min(pool, key=lambda p: (p.arrival, p.pid))   # <- replace this

    def quantum_expired(self, proc, clock):
        return False


# --- try it out ---
results, timeline = run(WORKLOAD, MyScheduler())
report(results, timeline, MyScheduler().name)

# --- check the rules ---
print()
validate_scheduler(MyScheduler)

## 9.5 What To Submit

A single notebook named `04_Scheduler_<LastName>.ipynb`, pushed to your
`DSCI2022-LastName` repository, containing:

1. **Your scheduler class**, commented well enough that a classmate could
   reimplement it from the comments alone.
2. **A passing `validate_scheduler()` run**, output visible.
3. **A comparison** against at least three of the six reference schedulers on the
   standard `WORKLOAD` — table plus Gantt charts.
4. **A workload you designed** that makes your scheduler look good, and **one that
   makes it look bad.** Both, with a sentence explaining why each behaves that way.
   *An answer with no failure case will not receive full credit.*
5. **A written analysis, 300–500 words**, covering:
   - What you optimised for, and what you consciously sacrificed
   - Every constant you chose and why (R8)
   - Whether your policy can starve a process, and under exactly what conditions
   - Where you would deploy this in the real world, and where you absolutely would not

## 9.6 Grading

| Component | Weight |
|---|---|
| Correctness — passes the validator and does what you claim | 30% |
| Rule compliance R1–R8 | 15% |
| Comparison and Gantt analysis | 20% |
| Good-case *and* bad-case workloads | 15% |
| Written analysis | 20% |

**Automatic deductions:** modifying `run()` or `Scheduler` (−20), non-deterministic
tie-breaking (−10), unexplained magic numbers (−5 each).

---

## 9.7 Things That Will Make Your Write-Up Better

- **Count the context switches.** A policy that wins on waiting time while switching
  three times as often may lose on real hardware. Add a per-switch cost to your
  analysis and see if your conclusion survives.
- **Vary the workload shape.** Many short jobs, a few long ones, everything arriving
  at once, arrivals spread out. Most schedulers have a workload that embarrasses them.
- **Look at the worst case, not just the average.** A scheduler with a good average
  and one process waiting 40 ticks is often unacceptable in practice. Report the
  maximum waiting time alongside the mean.
- **Connect it back to the lecture.** The reason your threaded Python program printed
  its output in a different order on every run is precisely this machinery, running
  in your operating system, making decisions you did not control.